# Phase 4: Leakage-Safe Machine-Learning Modeling

This notebook uses the Phase 3 predictor dataset to model calibrated 30% creatinine–cystatin C discordance.

It:

1. Loads `Cystatin_C_Phase3_Predictor_Outputs.zip`.
2. Excludes cystatin C and all cystatin C-derived fields from predictors.
3. Performs imputation and encoding inside each training pipeline.
4. Compares logistic regression, elastic net, random forest, and XGBoost.
5. Uses repeated stratified cross-validation.
6. Performs two cycle holdouts:
   - train on 1999–2000, test on 2001–2002;
   - train on 2001–2002, test on 1999–2000.
7. Reports AUROC, precision-recall AUC, Brier score, sensitivity, specificity, PPV, NPV, and testing-budget performance.
8. Saves all tables and predictions.

The primary outcome is `DISCORDANCE_30_CALIBRATED`.

In [ ]:
# Cell 1 — Install package
%pip -q install xgboost

In [ ]:
# Cell 2 — Imports and folders

import json
import zipfile
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

ROOT = Path("/content/cystatin_c_phase4")
INPUT = ROOT / "input"
OUTPUT = ROOT / "outputs"
INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print("✅ Environment ready")

In [ ]:
# Cell 3 — Upload and extract Phase 3 ZIP

from google.colab import files

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]

if len(zip_names) != 1:
    raise ValueError(
        "Upload exactly one ZIP file: "
        "Cystatin_C_Phase3_Predictor_Outputs.zip"
    )

zip_path = INPUT / zip_names[0]
zip_path.write_bytes(uploaded[zip_names[0]])

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(INPUT)

data_path = INPUT / "nhanes_final_predictor_dataset.csv"

if not data_path.exists():
    raise FileNotFoundError(
        "nhanes_final_predictor_dataset.csv was not found in the ZIP."
    )

df = pd.read_csv(data_path)

print("✅ Dataset loaded:", df.shape)
print(df["CYCLE"].value_counts())

In [ ]:
# Cell 4 — Define outcome and predictors

OUTCOME = "DISCORDANCE_30_CALIBRATED"
WEIGHT = "WTSCY4YR"

continuous_features = [
    "RIDAGEYR",
    "CREATININE_CALIBRATED",
    "LBXSBU",
    "LBXSAL",
    "LBXSGL",
    "BMI",
    "WAIST_CM",
    "MEAN_SBP",
    "MEAN_DBP",
    "HEMOGLOBIN",
    "CRP_MG_DL",
    "UACR_MG_G",
]

categorical_features = [
    "RIAGENDR",
    "RIDRETH1",
    "DIAGNOSED_DIABETES",
    "DIAGNOSED_HYPERTENSION",
    "SMOKING_STATUS",
    "ANY_CVD",
]

features = continuous_features + categorical_features

required = features + [OUTCOME, WEIGHT, "CYCLE", "SEQN"]
missing = [column for column in required if column not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

X = df[features].copy()
y = df[OUTCOME].astype(int).copy()
weights = pd.to_numeric(df[WEIGHT], errors="coerce").fillna(1.0)

print("Participants:", f"{len(df):,}")
print("Primary cases:", int(y.sum()))
print("Primary prevalence:", f"{100*y.mean():.2f}%")
print("Predictors:", len(features))

In [ ]:
# Cell 5 — Preprocessing and models

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, continuous_features),
    ("categorical", categorical_transformer, categorical_features),
])

models = {
    "Logistic regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(
            penalty="l2",
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
            random_state=RANDOM_STATE
        ))
    ]),
    "Elastic net": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            l1_ratio=0.5,
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
            random_state=RANDOM_STATE
        ))
    ]),
    "Random forest": Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_leaf=10,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),
    "XGBoost": Pipeline([
        ("preprocess", preprocessor),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),
}

print("✅ Pipelines created")

In [ ]:
# Cell 6 — Metrics and testing-budget functions

def classification_metrics(y_true, probability, threshold=0.5):
    prediction = (probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, prediction, labels=[0, 1]
    ).ravel()

    sensitivity = tp / (tp + fn) if tp + fn else np.nan
    specificity = tn / (tn + fp) if tn + fp else np.nan
    ppv = tp / (tp + fp) if tp + fp else np.nan
    npv = tn / (tn + fn) if tn + fn else np.nan

    return {
        "AUROC": roc_auc_score(y_true, probability),
        "AUPRC": average_precision_score(y_true, probability),
        "Brier": brier_score_loss(y_true, probability),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "PPV": ppv,
        "NPV": npv,
    }


def budget_performance(y_true, probability, budgets=(0.10, 0.20, 0.30, 0.50)):
    table = []
    y_array = np.asarray(y_true)
    p_array = np.asarray(probability)

    order = np.argsort(-p_array)
    total_cases = y_array.sum()

    for budget in budgets:
        n_test = max(1, int(np.ceil(len(y_array) * budget)))
        selected = order[:n_test]
        cases_found = y_array[selected].sum()

        table.append({
            "Testing budget (%)": 100 * budget,
            "People tested": n_test,
            "Cases detected": int(cases_found),
            "Case detection (%)":
                100 * cases_found / total_cases if total_cases else np.nan,
            "Positive yield (%)":
                100 * cases_found / n_test if n_test else np.nan,
            "Number needed to test":
                n_test / cases_found if cases_found else np.nan,
        })

    return pd.DataFrame(table)

In [ ]:
# Cell 7 — Repeated stratified cross-validation

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=RANDOM_STATE
)

cv_rows = []
oof_predictions = {
    model_name: np.zeros((len(df), 5))
    for model_name in models
}

for split_number, (train_idx, test_idx) in enumerate(cv.split(X, y)):
    repeat_number = split_number // 5

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    w_train = weights.iloc[train_idx]

    for model_name, pipeline in models.items():
        fitted = clone(pipeline)
        fitted.fit(
            X_train,
            y_train,
            model__sample_weight=w_train.to_numpy()
        )

        probability = fitted.predict_proba(X_test)[:, 1]
        oof_predictions[model_name][test_idx, repeat_number] = probability

        metrics = classification_metrics(y_test, probability)
        metrics.update({
            "Model": model_name,
            "Repeat": repeat_number + 1,
            "Fold": split_number % 5 + 1,
        })
        cv_rows.append(metrics)

cv_results = pd.DataFrame(cv_rows)

cv_summary = (
    cv_results.groupby("Model")
    .agg({
        "AUROC": ["mean", "std"],
        "AUPRC": ["mean", "std"],
        "Brier": ["mean", "std"],
        "Sensitivity": ["mean", "std"],
        "Specificity": ["mean", "std"],
        "PPV": ["mean", "std"],
        "NPV": ["mean", "std"],
    })
)

display(cv_summary.round(3))

In [ ]:
# Cell 8 — Aggregate repeated out-of-fold predictions

aggregated_predictions = pd.DataFrame({
    "SEQN": df["SEQN"],
    "CYCLE": df["CYCLE"],
    "Observed": y,
})

aggregate_metric_rows = []
budget_tables = []

for model_name in models:
    mean_probability = oof_predictions[model_name].mean(axis=1)
    aggregated_predictions[model_name] = mean_probability

    metrics = classification_metrics(y, mean_probability)
    metrics["Model"] = model_name
    aggregate_metric_rows.append(metrics)

    budget = budget_performance(y, mean_probability)
    budget.insert(0, "Model", model_name)
    budget_tables.append(budget)

aggregate_metrics = pd.DataFrame(aggregate_metric_rows)
budget_results = pd.concat(budget_tables, ignore_index=True)

print("Repeated-CV aggregate performance:")
display(aggregate_metrics.sort_values("AUROC", ascending=False).round(3))

print("\nTesting-budget performance:")
display(budget_results.round(2))

In [ ]:
# Cell 9 — Cycle holdout validation in both directions

cycle_rows = []
cycle_predictions = []

directions = [
    ("1999-2000", "2001-2002"),
    ("2001-2002", "1999-2000"),
]

for train_cycle, test_cycle in directions:
    train_mask = df["CYCLE"].eq(train_cycle)
    test_mask = df["CYCLE"].eq(test_cycle)

    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask]
    w_train = weights.loc[train_mask]

    X_test = X.loc[test_mask]
    y_test = y.loc[test_mask]

    for model_name, pipeline in models.items():
        fitted = clone(pipeline)
        fitted.fit(
            X_train,
            y_train,
            model__sample_weight=w_train.to_numpy()
        )

        probability = fitted.predict_proba(X_test)[:, 1]
        metrics = classification_metrics(y_test, probability)

        metrics.update({
            "Model": model_name,
            "Train cycle": train_cycle,
            "Test cycle": test_cycle,
            "Test N": int(test_mask.sum()),
            "Test cases": int(y_test.sum()),
        })
        cycle_rows.append(metrics)

        cycle_predictions.append(pd.DataFrame({
            "SEQN": df.loc[test_mask, "SEQN"].to_numpy(),
            "Observed": y_test.to_numpy(),
            "Probability": probability,
            "Model": model_name,
            "Train cycle": train_cycle,
            "Test cycle": test_cycle,
        }))

cycle_results = pd.DataFrame(cycle_rows)
cycle_prediction_df = pd.concat(cycle_predictions, ignore_index=True)

display(
    cycle_results.sort_values(
        ["Test cycle", "AUROC"],
        ascending=[True, False]
    ).round(3)
)

In [ ]:
# Cell 10 — Choose best model and fit on all data

best_model_name = (
    aggregate_metrics.sort_values("AUROC", ascending=False)
    .iloc[0]["Model"]
)

best_model = clone(models[best_model_name])
best_model.fit(
    X,
    y,
    model__sample_weight=weights.to_numpy()
)

print("Best model:", best_model_name)

In [ ]:
# Cell 11 — ROC and precision-recall curves

plt.figure(figsize=(7, 6))

for model_name in models:
    probability = aggregated_predictions[model_name]
    fpr, tpr, _ = roc_curve(y, probability)
    auc_value = roc_auc_score(y, probability)
    plt.plot(fpr, tpr, label=f"{model_name} (AUC={auc_value:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False-positive rate")
plt.ylabel("True-positive rate")
plt.title("Repeated Cross-Validated ROC Curves")
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(figsize=(7, 6))

for model_name in models:
    probability = aggregated_predictions[model_name]
    precision, recall, _ = precision_recall_curve(y, probability)
    ap_value = average_precision_score(y, probability)
    plt.plot(recall, precision, label=f"{model_name} (AP={ap_value:.3f})")

plt.axhline(y.mean(), linestyle="--", label=f"Prevalence={y.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Repeated Cross-Validated Precision–Recall Curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12 — Calibration table

best_probability = aggregated_predictions[best_model_name]

calibration_data = pd.DataFrame({
    "Observed": y,
    "Probability": best_probability
})

calibration_data["Decile"] = pd.qcut(
    calibration_data["Probability"],
    q=10,
    duplicates="drop"
)

calibration_table = (
    calibration_data.groupby("Decile", observed=True)
    .agg(
        N=("Observed", "size"),
        Mean_predicted_risk=("Probability", "mean"),
        Observed_prevalence=("Observed", "mean"),
    )
    .reset_index()
)

display(calibration_table.round(3))

plt.figure(figsize=(6, 6))
plt.plot(
    calibration_table["Mean_predicted_risk"],
    calibration_table["Observed_prevalence"],
    marker="o"
)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Mean predicted risk")
plt.ylabel("Observed prevalence")
plt.title(f"Calibration: {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13 — Permutation importance

importance = permutation_importance(
    best_model,
    X,
    y,
    scoring="roc_auc",
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance_table = pd.DataFrame({
    "Feature": features,
    "Importance mean": importance.importances_mean,
    "Importance SD": importance.importances_std,
}).sort_values("Importance mean", ascending=False)

display(importance_table.round(4))

In [ ]:
# Cell 14 — Save all outputs

cv_results_path = OUTPUT / "repeated_cv_fold_results.csv"
aggregate_metrics_path = OUTPUT / "repeated_cv_aggregate_metrics.csv"
predictions_path = OUTPUT / "repeated_cv_predictions.csv"
budget_path = OUTPUT / "testing_budget_results.csv"
cycle_results_path = OUTPUT / "cycle_holdout_results.csv"
cycle_predictions_path = OUTPUT / "cycle_holdout_predictions.csv"
calibration_path = OUTPUT / "best_model_calibration.csv"
importance_path = OUTPUT / "best_model_permutation_importance.csv"
model_path = OUTPUT / "best_model.joblib"

cv_results.to_csv(cv_results_path, index=False)
aggregate_metrics.to_csv(aggregate_metrics_path, index=False)
aggregated_predictions.to_csv(predictions_path, index=False)
budget_results.to_csv(budget_path, index=False)
cycle_results.to_csv(cycle_results_path, index=False)
cycle_prediction_df.to_csv(cycle_predictions_path, index=False)
calibration_table.to_csv(calibration_path, index=False)
importance_table.to_csv(importance_path, index=False)
joblib.dump(best_model, model_path)

summary = {
    "primary_outcome": OUTCOME,
    "participants": int(len(df)),
    "cases": int(y.sum()),
    "prevalence": float(y.mean()),
    "best_model": best_model_name,
}

(OUTPUT / "analysis_summary.json").write_text(
    json.dumps(summary, indent=2)
)

print("✅ All outputs saved")

In [ ]:
# Cell 15 — Download ZIP

output_zip = Path("/content/Cystatin_C_Phase4_Modeling_Outputs.zip")

with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT.iterdir():
        zf.write(path, arcname=path.name)

print("✅ Created:", output_zip)
files.download(str(output_zip))

## Upload next

Upload `Cystatin_C_Phase4_Modeling_Outputs.zip`.

The key files are:

- `repeated_cv_aggregate_metrics.csv`
- `cycle_holdout_results.csv`
- `testing_budget_results.csv`
- `best_model_permutation_importance.csv`